<a href="https://colab.research.google.com/github/bramefendi15-ai/machine-learning-zoomcamp-homework/blob/main/ML_ZOOMCZMP_HOMEORK_05_deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

python version

In [1]:
import sys
print(sys.version)

3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]


In [2]:
!git clone --depth 1 --filter=blob:none --sparse https://github.com/DataTalksClub/machine-learning-zoomcamp.git
%cd machine-learning-zoomcamp
!git sparse-checkout set cohorts/2026/homework/05-deployment
%cd cohorts/2026/homework/05-deployment
!ls -lah

Cloning into 'machine-learning-zoomcamp'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 138 (delta 0), reused 108 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 63.21 KiB | 2.11 MiB/s, done.
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 5 (delta 1), reused 1 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (5/5), 14.17 KiB | 14.17 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/machine-learning-zoomcamp
remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 20 (delta 0), reused 17 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (20/20), 23.85 KiB | 977.00 KiB/s, done.
/content/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment
tota

In [3]:
!cat homework.md

## Homework 5: Deploying Machine Learning Models

This homework uses a frozen lead-scoring release for the 2026 cohort. The
artifact, dataset checksum, training script, API, and container definition are
in this directory. No prior-cohort model or generated answer set is used.

The model was trained from
[`course_lead_scoring_2026.csv`](../../data/course_lead_scoring_2026.csv) with
the following observable features:

| Feature | Type | Missing-value rule |
| --- | --- | --- |
| `lead_source` | categorical | `NA` |
| `industry` | categorical | `NA` |
| `employment_status` | categorical | `NA` |
| `location` | categorical | `NA` |
| `number_of_courses_viewed` | integer | training-set median |
| `annual_income` | number | training-set median |
| `interaction_count` | integer | training-set median |
| `lead_score` | number | training-set median |

We use a `DictVectorizer` followed by
`LogisticRegression(solver="liblinear", C=1.0)`. The training script records
the data checksum and imputati

In [4]:
!cat .python-version
!cat pyproject.toml

3.11.15
[project]
name = "mlzoomcamp-2026-deployment"
version = "0.1.0"
description = "Reference deployment project for ML Zoomcamp 2026"
requires-python = ">=3.11,<3.12"
dependencies = [
    "fastapi==0.119.1",
    "numpy==2.3.3",
    "scikit-learn==1.7.2",
    "uvicorn==0.38.0",
]

[tool.uv]
exclude-newer = "2026-09-14T23:59:59Z"


In [7]:
%cd /content/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment

!pip -q install uv

!uv python install 3.11.15
!uv venv --python 3.11.15 .venv
!uv pip install --python .venv/bin/python -r pyproject.toml

/content/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment
Installed Python 3.11.15 in 1.71s
 + cpython-3.11.15-linux-x86_64-gnu (python3.11)
Using CPython 3.11.15
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate
Resolved 18 packages in 637ms
Prepared 18 packages in 1.72s
Installed 18 packages in 77ms
 + annotated-types==0.8.0
 + anyio==4.15.1
 + click==8.5.0
 + cloudpickle==3.1.2
 + fastapi==0.119.1
 + h11==0.16.0
 + idna==3.19
 + joblib==1.6.0
 + numpy==2.3.3
 + pydantic==2.13.5
 + pydantic-core==2.46.5
 + scikit-learn==1.7.2
 + scipy==1.17.1
 + starlette==0.48.0
 + threadpoolctl==3.6.0
 + typing-extensions==4.16.0
 + typing-inspection==0.4.4
 + uvicorn==0.38.0


In [8]:
!uv run --python .venv/bin/python python -c "import sys, numpy, sklearn; print(sys.version); print('NumPy:', numpy.__version__); print('Scikit-learn:', sklearn.__version__)"

3.11.15 (main, Aug  7 2026, 02:25:55) [Clang 22.1.3 ]
NumPy: 2.3.3
Scikit-learn: 1.7.2


In [9]:
%cd /content/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment
!uv --version

/content/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment
uv 0.13.5 (x86_64-unknown-linux-gnu)


In [13]:
%%writefile /content/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment/calculate.py
import pickle

with open("/content/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment/pipeline.bin", "rb") as f:
    pipeline = pickle.load(f)

print("Model loaded successfully!")


Writing /content/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment/calculate.py


In [14]:
!cd /content/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment && uv run --python .venv/bin/python python calculate.py

Model loaded successfully!


In [15]:
%cd /content/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment

!cat predict.py
!cat model.py

/content/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment
from __future__ import annotations

import hashlib
import os
import pickle
from pathlib import Path
from typing import Any

from fastapi import FastAPI
from pydantic import BaseModel

from model import DEFAULTS_PATH, normalize_record


MODEL_PATH = Path(os.getenv("MODEL_PATH", Path(__file__).with_name("pipeline.bin")))
with MODEL_PATH.open("rb") as model_file:
    pipeline = pickle.load(model_file)
MODEL_SHA256 = hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()
FEATURE_DEFAULTS_SHA256 = hashlib.sha256(DEFAULTS_PATH.read_bytes()).hexdigest()

app = FastAPI(title="ML Zoomcamp 2026 lead scoring")


class Lead(BaseModel):
    lead_source: str | None = None
    industry: str | None = None
    employment_status: str | None = None
    location: str | None = None
    number_of_courses_viewed: int | None = None
    annual_income: float | None = None
    interaction_count: int | None = None
    lead_score: float | None = N

In [16]:
%%writefile /content/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment/calculate.py
import pickle
from pathlib import Path
from model import normalize_record

base = Path(__file__).parent

with open(base / "pipeline.bin", "rb") as f:
    pipeline = pickle.load(f)

leads = {
    "Q3": {
        "lead_source": "paid_ads",
        "industry": "technology",
        "employment_status": "employed",
        "location": "north_america",
        "number_of_courses_viewed": 2,
        "annual_income": 79276.0,
        "interaction_count": 4,
        "lead_score": 0.41,
    },
    "Q4": {
        "lead_source": "organic_search",
        "industry": "technology",
        "employment_status": "employed",
        "location": "europe",
        "number_of_courses_viewed": 4,
        "annual_income": 80304.0,
        "interaction_count": 7,
        "lead_score": 0.74,
    },
}

for question, lead in leads.items():
    record = normalize_record(lead)
    probability = pipeline.predict_proba([record])[0, 1]
    print(f"{question}: {probability:.3f}")

Overwriting /content/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment/calculate.py


In [17]:
%cd /content/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment
!uv run --python .venv/bin/python python calculate.py

/content/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment
Q3: 0.533
Q4: 0.770


In [18]:
!docker --version

/bin/bash: line 1: docker: command not found


In [19]:
%cd /content/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment

import hashlib
import json
from pathlib import Path

model_path = Path("pipeline.bin")
metadata = json.loads(Path("model_metadata.json").read_text())

print("Model SHA256:", hashlib.sha256(model_path.read_bytes()).hexdigest())
print("Metadata:")
print(json.dumps(metadata, indent=2))

/content/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment
Model SHA256: 1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1
Metadata:
{
  "dataset": "course_lead_scoring_2026.csv",
  "dataset_sha256": "2d9da196bdefd2a45aa0a17ca21c19f06a5d9d56af628be1f04bbb3531cc12e9",
  "rows": 5000,
  "features": [
    "lead_source",
    "industry",
    "employment_status",
    "location",
    "number_of_courses_viewed",
    "annual_income",
    "interaction_count",
    "lead_score"
  ],
  "target": "converted",
  "model": "DictVectorizer + LogisticRegression(solver=liblinear, C=1.0)",
  "random_state": 42,
  "scikit_learn": "1.7.2",
  "numeric_imputation": "training-set median",
  "numeric_defaults": {
    "number_of_courses_viewed": 2.0,
    "annual_income": 55051.0,
    "interaction_count": 4.0,
    "lead_score": 0.49
  },
  "model_sha256": "1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1",
  "feature_defaults_sha256": "1694f842b908fac9f53c0cc19be6d91f934b

In [20]:
%cd /content/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment

!nohup uv run --python .venv/bin/python uvicorn predict:app --host 0.0.0.0 --port 9696 > /tmp/mlzoomcamp_api.log 2>&1 &

/content/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment


In [21]:
!sleep 2
!cat /tmp/mlzoomcamp_api.log
!curl -s http://localhost:9696/health

INFO:     Started server process [9729]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:9696 (Press CTRL+C to quit)
{"status":"ok","model_sha256":"1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1","feature_defaults_sha256":"1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd"}

In [23]:
!curl -s -X POST http://localhost:9696/predict \
  -H "Content-Type: application/json" \
  -d '{
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304,
    "interaction_count": 7,
    "lead_score": 0.74
  }'

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 10)

In [24]:
!curl -s -X POST http://localhost:9696/predict -H "Content-Type: application/json" -d '{"lead_source":"organic_search","industry":"technology","employment_status":"employed","location":"europe","number_of_courses_viewed":4,"annual_income":80304,"interaction_count":7,"lead_score":0.74}'

{"conversion_probability":0.769799,"conversion":true}

In [25]:
! .venv/bin/python q6_test.py

{'conversion_probability': 0.769799, 'conversion': True}
